# 🎬 YT Short Clipper Pro — 1-Click Colab

Run everything in ONE cell. Just click ▶️ and wait for the Ngrok URL.

**Features**: AI Analysis, Face Tracking, Karaoke Subtitle, B-Roll Overlay, Background Music

**Setup**: Add `NGROK_AUTH_TOKEN` and API keys (`GEMINI_API_KEY` / `GROQ_API_KEY` / `OPENROUTER_API_KEY`) to Colab Secrets (🔑 sidebar) — or skip for manual input.

In [ ]:
#@title 🚀 1-Click Setup & Launch (Just run this cell) { display-mode: "form" }
#@markdown 1. Mount Google Drive
#@markdown 2. Install all dependencies
#@markdown 3. Clone repo & setup directories
#@markdown 4. Get API keys from Colab Secrets (or manual input)
#@markdown 5. Start Ngrok tunnel + Streamlit server
#@markdown 6. Show public URL for browser access

import subprocess, time, os, sys, getpass
from pathlib import Path

print("="*60)
print("🎬 YT Short Clipper Pro — 1-Click Launch")
print("="*60)

# --- Step 1: Mount Google Drive ---
print("\n[1/6] Mounting Google Drive...")
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("  ✅ Drive mounted!")
except Exception as e:
    print(f"  ⚠️ Drive mount failed: {e} — using local output only")

# --- Step 2: Install dependencies ---
print("\n[2/6] Installing dependencies...")
subprocess.run(["apt-get", "-qq", "install", "ffmpeg"], capture_output=True)

deps = [
    "yt-dlp[default]", "opencv-python-headless", "numpy", "Pillow",
    "requests", "mediapipe", "python-dotenv", "faster-whisper",
    "google-genai", "streamlit", "pyngrok"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + deps, capture_output=True)
print("  ✅ All dependencies installed!")

# --- Step 3: Clone repo ---
print("\n[3/6] Cloning repository...")
repo_dir = "/content/yt-short-clipper-offline"
if not os.path.exists(repo_dir):
    subprocess.run(["git", "clone", "https://github.com/Chukie99/yt-short-clipper-offline.git", repo_dir],
                   capture_output=True)
sys.path.insert(0, repo_dir)
os.chdir(repo_dir)

# Verify files
for f in ["clipper_core.py", "app.py", "requirements.txt"]:
    status = "✅" if os.path.exists(os.path.join(repo_dir, f)) else "❌"
    print(f"  {status} {f}")

# --- Step 4: Setup directories ---
print("\n[4/6] Setting up directories...")
from clipper_core import setup_directories
setup_directories(
    temp_dir="/content/temp",
    output_dir="/content/drive/MyDrive/YTShortClipper/output",
    config_file="/content/drive/MyDrive/YTShortClipper/config.json",
)
print("  ✅ Directories ready!")

# --- Step 5: Get API keys ---
print("\n[5/6] Loading API keys...")

# Try Colab Secrets first
secrets_loaded = False
try:
    from google.colab import userdata
    for key in ["GEMINI_API_KEY", "GROQ_API_KEY", "OPENROUTER_API_KEY", "NGROK_AUTH_TOKEN"]:
        val = userdata.get(key)
        if val:
            os.environ[key] = val
            secrets_loaded = True
            print(f"  ✅ {key} loaded from Secrets")
except Exception:
    pass

# Fallback to getpass
if not secrets_loaded:
    print("  ℹ️ No Colab Secrets found. Enter keys manually (skip with Enter):")
    for key in ["GEMINI_API_KEY", "GROQ_API_KEY", "OPENROUTER_API_KEY", "NGROK_AUTH_TOKEN"]:
        if not os.environ.get(key):
            label = key.replace("_", " ").title()
            val = getpass.getpass(f"  {label}: ")
            if val:
                os.environ[key] = val

# Verify
has_api = any(os.environ.get(k) for k in ["GEMINI_API_KEY", "GROQ_API_KEY", "OPENROUTER_API_KEY"])
has_ngrok = bool(os.environ.get("NGROK_AUTH_TOKEN"))
print(f"  {'✅' if has_api else '⚠️'} API Keys: {'Ready' if has_api else 'Missing (AI analysis disabled)'}")
print(f"  {'✅' if has_ngrok else '⚠️'} Ngrok: {'Ready' if has_ngrok else 'Missing (using Colab direct)'}")

# --- Step 6: Launch Streamlit + Ngrok ---
print("\n[6/6] Launching Streamlit server...")

# Kill any existing streamlit
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

# Start Streamlit in background
process = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port=8501",
     "--server.headless=true",
     "--server.address=0.0.0.0",
     "--browser.gatherUsageStats=false"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait for Streamlit to start
print("  ⏳ Waiting for Streamlit to start...")
for i in range(20):
    time.sleep(1)
    if process.poll() is not None:
        print(f"  ❌ Streamlit crashed! Exit code: {process.returncode}")
        break
else:
    print("  ✅ Streamlit is running!")

# Show public URL
print("\n" + "="*60)
ngrok_token = os.environ.get("NGROK_AUTH_TOKEN", "")
if ngrok_token:
    try:
        from pyngrok import ngrok, conf
        conf.get_default().auth_token = ngrok_token
        ngrok.kill()
        tunnel = ngrok.connect(8501, "http")
        public_url = tunnel.public_url
        print(f"🌐 PUBLIC URL (Ngrok): {public_url}")
    except Exception as e:
        print(f"⚠️ Ngrok error: {e}")
        print("🌐 Colab Direct: http://localhost:8501")
else:
    print("🌐 Colab Direct Link: http://localhost:8501")

print("="*60)
print("\nOpen the URL above in your browser to access the WebUI!")
print("To stop: run 'pkill -f streamlit' in a new cell\n")